# Pandas: Column Creation, Updating & Deletion
### A step-by-step, beginner-friendly module using transaction and customer data

Work through each section in order — later sections build on earlier ones.

---

## 📋 Module Contents

- **00** — Setup & Imports
- **01** — Load the Data
- **02** — Inspect the Data
- **03** — Work on a Copy
- **04** — Calculated Columns (Arithmetic)
- **05** — Flag Columns (Conditions)
- **06** — Update Values with .loc
- **07** — Multiple Conditions with .loc
- **08** — Multiple Categories with np.select
- **09** — Custom Functions + apply()
- **10** — Lambda Functions
- **11** — Vectorized Alternatives: np.where & .assign()
- **12** — Group-Based Columns — groupby().transform()
- **13** — Percentage Share Within a Group
- **14** — Rank Within a Group
- **15** — Cumulative Columns
- **16** — Merging Tables
- **17** — Columns from Merged Data
- **18** — Dummy Variables
- **19** — replace() and map()
- **20** — Delete Columns — .drop(), del, pop()
- **21** — Drop Rows — Filter, dropna()
- **22** — Select Columns to Keep
- **23** — Rename Columns
- **24** — Mini Case Study — Executive Dashboard
- **25** — Additional Classroom Practice Exercises
- **26** — Common Beginner Mistakes
- **27** — Quick Reference Cheat Sheet

---
## Section 00: Setup & Imports

Every notebook starts with imports. We need pandas for data manipulation and numpy for math operations.

In [1]:
# Python — run this first in every session
import pandas as pd
import numpy as np

# Show more columns in output (useful when DataFrame is wide)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

> ℹ️ **Why these display options?**
>
> By default, pandas hides columns if the table is wide. max_columns=50 lets us see up to 50 columns without truncation — useful when we create many new columns.

---
## Section 01: Load the Data

We work with two CSV files:

- transactions_advanced.csv — one row per order (transaction-level data)
- customers_advanced.csv — one row per customer (customer-level data)

In [2]:
transactions = pd.read_csv("transactions_advanced.csv")
customers    = pd.read_csv("customers_advanced.csv")

transactions.head()  # Show first 5 rows

,order_id,customer_id,order_date,country,channel,category,segment,payment_method,ship_method,quantity,unit_price,discount_rate,shipping_fee,status,satisfaction_score
0,ORD10228,C1013,2025-05-18,UK,Marketplace,Toys,Consumer,Credit Card,Express,5,60.26,NaN,0.00,Returned,4.0
1,ORD10194,C1022,2025-04-11,USA,Partner,Clothing,Consumer,Gift Card,Standard,1,67.94,0.15,12.99,Completed,4.0
2,ORD10088,C1033,2025-05-30,France,Marketplace,Toys,Consumer,Credit Card,Standard,3,62.34,0.10,12.99,Completed,3.0
3,ORD10095,C1034,2025-03-11,USA,Retail,Books,Enterprise,Credit Card,Standard,1,56.29,0.00,0.00,Cancelled,5.0
4,ORD10214,C1032,2025-06-03,UK,Online,Home,Consumer,Gift Card,Standard,7,102.61,0.00,4.99,Completed,4.0


In [3]:
customers.head()

,customer_id,signup_date,loyalty_tier,marketing_opt_in
0,C1000,2024-09-22,Gold,Yes
1,C1001,2025-01-19,Gold,No
2,C1002,2024-06-09,Silver,Yes
3,C1003,2024-08-04,Silver,Yes
4,C1004,2025-01-26,Bronze,Yes


> 💡 **Teaching Note**
>
> Ask students: "How many rows are in each table? How are they related?" A transaction has one customer_id — many transactions can belong to the same customer.

---
## Section 02: Inspect the Data

Before creating or changing any columns, always inspect the data. This prevents mistakes later.

> ℹ️ **The Four Inspection Steps**

1. Column names and data types
2. Missing values — which columns have gaps?
3. Summary statistics — what are the ranges?
4. First few rows — does the data look as expected?

In [4]:
transactions.info()             # Column names, data types, non-null counts

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            240 non-null    str    
 1   customer_id         240 non-null    str    
 2   order_date          240 non-null    str    
 3   country             235 non-null    str    
 4   channel             235 non-null    str    
 5   category            240 non-null    str    
 6   segment             240 non-null    str    
 7   payment_method      235 non-null    str    
 8   ship_method         240 non-null    str    
 9   quantity            240 non-null    int64  
 10  unit_price          236 non-null    float64
 11  discount_rate       234 non-null    float64
 12  shipping_fee        240 non-null    float64
 13  status              240 non-null    str    
 14  satisfaction_score  229 non-null    float64
dtypes: float64(4), int64(1), str(10)
memory usage: 28.3 KB


In [5]:
transactions.isna().sum()       # Count missing values per column

order_id               0
customer_id            0
order_date             0
country                5
channel                5
category               0
segment                0
payment_method         5
ship_method            0
quantity               0
unit_price             4
discount_rate          6
shipping_fee           0
status                 0
satisfaction_score    11
dtype: int64

In [6]:
transactions.describe(include="all")  # Summary stats for all columns

,order_id,customer_id,order_date,country,channel,category,segment,payment_method,ship_method,quantity,unit_price,discount_rate,shipping_fee,status,satisfaction_score
count,240,240,240,235,235,240,240,235,240,240.000000,236.000000,234.000000,240.000000,240,229.000000
unique,240,68,133,8,4,7,3,4,3,NaN,NaN,NaN,NaN,3,NaN
top,ORD10228,C1061,2025-02-27,USA,Online,Home,Consumer,Credit Card,Standard,NaN,NaN,NaN,NaN,Completed,NaN
freq,1,12,6,77,92,38,158,127,168,NaN,NaN,NaN,NaN,157,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.762500,79.685636,0.066453,6.629667,NaN,3.938865
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.414619,36.761416,0.073658,5.399738,NaN,1.133896
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.000000,17.130000,0.000000,0.000000,NaN,1.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,54.057500,0.000000,4.990000,NaN,4.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,71.515000,0.050000,4.990000,NaN,4.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.000000,102.310000,0.100000,7.990000,NaN,5.000000


---
## Section 03: Work on a Copy

When transforming data, it is safer to keep the raw data unchanged and work on a copy .

In [7]:
# Python — best practice
df = transactions.copy()
df.head()

,order_id,customer_id,order_date,country,channel,category,segment,payment_method,ship_method,quantity,unit_price,discount_rate,shipping_fee,status,satisfaction_score
0,ORD10228,C1013,2025-05-18,UK,Marketplace,Toys,Consumer,Credit Card,Express,5,60.26,NaN,0.00,Returned,4.0
1,ORD10194,C1022,2025-04-11,USA,Partner,Clothing,Consumer,Gift Card,Standard,1,67.94,0.15,12.99,Completed,4.0
2,ORD10088,C1033,2025-05-30,France,Marketplace,Toys,Consumer,Credit Card,Standard,3,62.34,0.10,12.99,Completed,3.0
3,ORD10095,C1034,2025-03-11,USA,Retail,Books,Enterprise,Credit Card,Standard,1,56.29,0.00,0.00,Cancelled,5.0
4,ORD10214,C1032,2025-06-03,UK,Online,Home,Consumer,Gift Card,Standard,7,102.61,0.00,4.99,Completed,4.0


> ⚠️ **Why .copy() matters**
>
> Without .copy() , changes to df may also change transactions — the original data. With .copy() , you are working on a completely independent DataFrame. You can always restart from transactions if something goes wrong.

---
## Section 04: Calculated Columns — Arithmetic

The most common task: create a new column by doing arithmetic on existing columns.

> 💡 **Pattern**
>
> df["new_column"] = df["col_a"] + df["col_b"]

### Concept Build columns step by step

Always handle missing values before doing arithmetic. Here, discount_rate has some missing values, so we fill them with 0 first.

In [8]:
# Python — step-by-step column creation
# Step 1: Clean the discount_rate column first
df["discount_rate_clean"] = df["discount_rate"].fillna(0)

# Step 2: Gross sales = quantity × unit price
df["gross_sales"] = df["quantity"] * df["unit_price"]

# Step 3: Discount amount = gross sales × discount rate
df["discount_amount"] = df["gross_sales"] * df["discount_rate_clean"]

# Step 4: Net sales = gross - discount + shipping
df["net_sales"] = (
    df["gross_sales"]
    - df["discount_amount"]
    + df["shipping_fee"]
)

df[["gross_sales", "discount_amount", "shipping_fee", "net_sales"]].head()

,gross_sales,discount_amount,shipping_fee,net_sales
0,301.30,0.000,0.00,301.300
1,67.94,10.191,12.99,70.739
2,187.02,18.702,12.99,181.308
3,56.29,0.000,0.00,56.290
4,718.27,0.000,4.99,723.260


> ℹ️ **Why create discount_rate_clean?**
>
> The original discount_rate has missing values. Instead of overwriting it, we create a clean version first. This keeps the raw data intact and makes the transformation transparent.

### ✏️ Exercise 1

Create a column called discount_rate_pct that shows the discount rate as a percentage (0–100 scale, not 0–1).
Formula: discount_rate_clean × 100
Then show the first 5 rows of discount_rate_clean and discount_rate_pct .

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [10]:
# Solution
df["discount_rate_pct"] = (df["discount_rate_clean"] * 100).astype(int)

df[["discount_rate_clean", "discount_rate_pct"]].head()

,discount_rate_clean,discount_rate_pct
0,0.00,0
1,0.15,15
2,0.10,10
3,0.00,0
4,0.00,0


---
## Section 05: Flag Columns — Boolean Conditions

A flag column marks whether a condition is True or False for each row. Very common in business analysis.

### Example True/False flag

In [11]:
# Flag: is this a high-value order?
df["high_value_order"] = df["net_sales"] > 200

df[["net_sales", "high_value_order"]].head()

,net_sales,high_value_order
0,301.300,True
1,70.739,False
2,181.308,False
3,56.290,False
4,723.260,True


### Example 0/1 flag using .astype(int)

Sometimes we want 1 and 0 instead of True and False . Add .astype(int) .

In [16]:
# def is_problem_order(row):
#     if row["status"] in ["Returned", "Cancelled"] and row["net_sales"] > 0:
#         return True
#     else:
#         return False

# problem_order_condition = df.apply(is_problem_order, axis=1)

In [17]:
df["status"].isin(["Returned", "Cancelled"])

0       True
1      False
2      False
3       True
4      False
       ...  
235    False
236    False
237     True
238     True
239     True
Name: status, Length: 240, dtype: bool

In [12]:
# Flag returned or cancelled orders as 1, others as 0
df["is_return_or_cancelled"] = (
    df["status"].isin(["Returned", "Cancelled"])
).astype(int)

df[["status", "is_return_or_cancelled"]].head(10)

,status,is_return_or_cancelled
0,Returned,1
1,Completed,0
2,Completed,0
3,Cancelled,1
4,Completed,0
5,Completed,0
6,Completed,0
7,Completed,0
8,Completed,0
9,Cancelled,1


### ✏️ Exercise 2

Create a column called has_discount .
1 if discount_rate_clean > 0 0 otherwise
Show discount_rate_clean and has_discount for the first 5 rows.

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [18]:
# Solution
df["has_discount"] = (df["discount_rate_clean"] > 0).astype(int)

df[["discount_rate_clean", "has_discount"]].head()

,discount_rate_clean,has_discount
0,0.00,0
1,0.15,1
2,0.10,1
3,0.00,0
4,0.00,0


---
## Section 06: Update Values with .loc

.loc is one of the most important pandas tools. It lets you view or update specific rows and columns .

> ℹ️ **The .loc Pattern**
>
> df.loc[condition, "column_name"] = new_value Read it as: "In rows where the condition is true, set this column to this value."

### Example 1 View rows before changing them

Good habit: always look at the rows you are about to change first.

In [19]:
# Define the condition
problem_orders = df["status"].isin(["Returned", "Cancelled"])

# View those rows before changing anything
df.loc[problem_orders, ["order_id", "status", "net_sales"]].head(2)

,order_id,status,net_sales
0,ORD10228,Returned,301.30
3,ORD10095,Cancelled,56.29


### Example 2 Update one column

Set net_sales to 0 for returned or cancelled orders.

In [ ]:
df.loc[problem_orders, "net_sales"] = 0

# Check the result
df.loc[problem_orders, ["order_id", "status", "net_sales"]].head(10)

### Example 3 Create a column, then update selected rows

A very beginner-friendly pattern: set a default for all rows, then override specific ones.

In [20]:
# Step 1: Default value for all rows
df["order_status_group"] = "Completed Order"

# Step 2: Override for returned/cancelled rows
df.loc[
    df["status"].isin(["Returned", "Cancelled"]),
    "order_status_group"
] = "Problem Order"

df[["status", "order_status_group"]].head(2)

,status,order_status_group
0,Returned,Problem Order
1,Completed,Completed Order


### Example 4 Fix missing values with .loc

In [ ]:
# Count missing values before
print("Missing country values:", df["country"].isna().sum())

# Fill missing country with "Unknown"
df.loc[df["country"].isna(), "country"] = "Unknown"

print("After fix:", df["country"].isna().sum())

### Example 5 Update multiple columns at once

In [ ]:
# For cancelled orders, update two columns simultaneously
df.loc[
    df["status"] == "Cancelled",
    ["net_sales", "order_status_group"]
] = [0, "Problem Order"]

> ⚠️ **⚠ Common Beginner Mistake — Chained Assignment**
>
> Do NOT do this: # WRONG — may silently fail to update the original DataFrame df[df[ "status" ] == "Cancelled" ][ "net_sales" ] = 0 Use .loc instead: # CORRECT df.loc[df[ "status" ] == "Cancelled" , "net_sales" ] = 0

### ✏️ Exercise 3

Create a column called review_needed .
1 Set the default value to "No" for all rows 2 Use .loc to change it to "Yes" when status is "Returned" or "Cancelled" 3 Display order_id , status , and review_needed for the first 10 rows

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [21]:
# Solution
df["review_needed"] = "No"

df.loc[
    df["status"].isin(["Returned", "Cancelled"]),
    "review_needed"
] = "Yes"

df[["order_id", "status", "review_needed"]].head(10)

,order_id,status,review_needed
0,ORD10228,Returned,Yes
1,ORD10194,Completed,No
2,ORD10088,Completed,No
3,ORD10095,Cancelled,Yes
4,ORD10214,Completed,No
5,ORD10004,Completed,No
6,ORD10093,Completed,No
7,ORD10027,Completed,No
8,ORD10170,Completed,No
9,ORD10237,Cancelled,Yes


---
## Section 07: Multiple Conditions with .loc

> ℹ️ **Rules for Multiple Conditions**
>
> Use & for AND (both conditions must be true) Use | for OR (at least one condition must be true) Always wrap each condition in its own parentheses ( )

### Example — AND condition

In [22]:
# VIP order = US country AND net_sales >= 300
df["vip_order"] = 0

df.loc[
    (df["country"] == "US") & (df["net_sales"] >= 300),
    "vip_order"
] = 1

df[["country", "net_sales", "vip_order"]].head(10)

,country,net_sales,vip_order
0,UK,301.300,0
1,USA,70.739,0
2,France,181.308,0
3,USA,56.290,0
4,UK,723.260,0
5,UK,353.167,0
6,USA,66.429,0
7,Brazil,539.190,0
8,USA,725.590,0
9,Australia,159.502,0


### ✏️ Exercise 4

Create a column called priority_review .
Set it to 1 when:
net_sales > 200 , OR status is "Returned" or "Cancelled"
Otherwise set it to 0 .

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [23]:
# Solution
df["priority_review"] = 0

df.loc[
    (df["net_sales"] > 200) | (df["status"].isin(["Returned", "Cancelled"])),
    "priority_review"
] = 1

df[["net_sales", "status", "priority_review"]].head(10)

,net_sales,status,priority_review
0,301.300,Returned,1
1,70.739,Completed,0
2,181.308,Completed,0
3,56.290,Cancelled,1
4,723.260,Completed,1
5,353.167,Completed,1
6,66.429,Completed,0
7,539.190,Completed,1
8,725.590,Completed,1
9,159.502,Cancelled,1


---
## Section 08: Multiple Categories with np.select

When you have several different categories to assign, repeated .loc works but np.select() is cleaner and easier to read.

> ℹ️ **np.select Pattern**
>
> List all conditions (in order of priority) List the matching output values Set a default for rows that match none of the conditions

### Example — Order size categories

In [ ]:
conditions = [
    df["net_sales"] >= 300,                            # First check
    df["net_sales"].between(100, 299.99),              # Second check
    df["net_sales"].between(0.01, 99.99),             # Third check
    df["net_sales"] == 0                               # Fourth check
]

choices = ["Large", "Medium", "Small", "No Revenue"]

df["order_size"] = np.select(conditions, choices, default="Unknown")

df[["net_sales", "order_size"]].head(10)

### ✏️ Exercise 5

Create a column called discount_group using np.select() :
"No Discount" if discount_rate_clean == 0 "Small Discount" if discount_rate_clean > 0 and <= 0.10 "Large Discount" if discount_rate_clean > 0.10

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [ ]:
# Solution
conditions = [
    df["discount_rate_clean"] == 0,
    (df["discount_rate_clean"] > 0) & (df["discount_rate_clean"] <= 0.10),
    df["discount_rate_clean"] > 0.10
]

choices = ["No Discount", "Small Discount", "Large Discount"]

df["discount_group"] = np.select(conditions, choices, default="Unknown")

df[["discount_rate_clean", "discount_group"]].head(10)

---
## Section 09: Custom Functions + apply()

When the logic is too complex for one line, define a custom function and apply it row by row.

### Example — Risk level classification

In [24]:
def assign_risk_level(row):
    """Return a risk label for each order."""
    if row["status"] in ["Returned", "Cancelled"]:
        return "High Risk"
    elif row["payment_method"] == "Wire" and row["net_sales"] > 250:
        return "High Risk"
    elif row["discount_rate_clean"] >= 0.3:
        return "Medium Risk"
    else:
        return "Low Risk"

df["risk_level"] = df.apply(assign_risk_level, axis=1)

df[["status", "payment_method", "net_sales", "risk_level"]].head(2)

,status,payment_method,net_sales,risk_level
0,Returned,Credit Card,301.300,High Risk
1,Completed,Gift Card,70.739,Low Risk


> ℹ️ **What does axis=1 mean?**
>
> axis=1 → apply the function row by row (each call receives one row) axis=0 (default) → apply column by column Inside the function, row["net_sales"] refers to the value in that specific row.

> 💡 **Beginner Advice**
>
> Use apply(axis=1) only when the logic is complex and involves multiple columns. For simple arithmetic or single-column conditions, use direct pandas operations — they are faster.

### ✏️ Exercise 6

Write a function called shipping_label(row) . Rules:
If shipping_fee == 0 → return "Free Shipping" If shipping_fee > 0 and net_sales >= 100 → return "Paid Shipping - High Value" Otherwise → return "Paid Shipping"
Apply it to create a column called shipping_label .

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [26]:
# Solution
def shipping_label(row):
    if row["shipping_fee"] == 0:
        return "Free Shipping"
    elif (row["shipping_fee"] > 0) and (row["net_sales"] >= 100):
        return "Paid Shipping - High Value"
    else:
        return "Paid Shipping"

df["shipping_label"] = df.apply(shipping_label, axis=1)

df[["shipping_fee", "net_sales", "shipping_label"]].head(2)

,shipping_fee,net_sales,shipping_label
0,0.00,301.300,Free Shipping
1,12.99,70.739,Paid Shipping


---
## Section 10: Lambda Functions

A lambda is a short, one-line function. Useful for simple transformations that don't need a full def .

In [27]:
# Create a region column from country using lambda
df["region"] = df["country"].apply(
    lambda x: "North America" if x in ["USA", "Canada"] else "International"
)

df[["country", "region"]].head(2)

,country,region
0,UK,International
1,USA,North America


> ⚠️ **Caution**
>
> Use lambda for simple, readable logic. If the logic needs more than one line, write a named def function instead — it will be easier to read and debug.

---
## Section 11: Vectorized Alternatives: np.where & .assign()

For simple if/else logic, np.where() is faster and cleaner than apply() .

> ℹ️ **np.where Pattern**
>
> np.where(condition, value_if_true, value_if_false)

In [28]:
# Python — np.where vs apply
# Cleaner and faster than apply(lambda x: ...)
df["region_v2"] = np.where(
    df["country"].isin(["USA", "Canada"]),
    "North America",
    "International"
)

df[["country", "region", "region_v2"]].head(2)

,country,region,region_v2
0,UK,International,International
1,USA,North America,North America


### Concept .assign() for clean pipelines

.assign() creates new columns and is ideal for writing a clean pipeline. The lambda x: inside refers to the DataFrame at that step.

In [30]:
# .assign() is great for readable multi-step pipelines
df = df.assign(
    revenue_after_return=lambda x: np.where(
        x["is_return_or_cancelled"] == 1, 0, x["net_sales"]
    )
)

df[["net_sales", "is_return_or_cancelled", "revenue_after_return"]].head(2)

,net_sales,is_return_or_cancelled,revenue_after_return
0,301.300,1,0.000
1,70.739,0,70.739


---
## Section 12: Group-Based Columns — groupby().transform()

One of the most powerful beginner skills: create a column whose value depends on the group the row belongs to.

Examples of group-based questions:

- What is the average order value in this customer's country ?
- Is this order above or below average for its category ?
- What is this order's rank within its channel ?

> ℹ️ **Why transform() and not agg()?**
>
> agg() → returns one row per group (a summary table) transform() → returns one value per original row — so you can assign it back as a new column

### Example — Country average sales

In [33]:
df.groupby("country")["net_sales"].transform("mean")

0      289.659554
1      239.955954
2      373.793609
3      239.955954
4      289.659554
          ...    
235    373.793609
236    236.023812
237    239.955954
238    373.793609
239    284.493140
Name: net_sales, Length: 240, dtype: float64

In [32]:
# Attach the average net_sales for each country to every row
df["country_avg_sales"] = (
    df
    .groupby("country")["net_sales"]
    .transform("mean")
)

# Now compare each row to its country average
df["above_country_average"] = (
    df["net_sales"] > df["country_avg_sales"]
).astype(int)

df[["country", "net_sales", "country_avg_sales", "above_country_average"]].head(2)

,country,net_sales,country_avg_sales,above_country_average
0,UK,301.300,289.659554,1
1,USA,70.739,239.955954,0


### ✏️ Exercise 7

Create a column called category_avg_sales containing the average net_sales for each row's category .
Then create above_category_average = 1 if the row's net_sales is above its category average.

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [34]:
# Solution
df["category_avg_sales"] = (
    df.groupby("category")["net_sales"].transform("mean")
)

df["above_category_average"] = (
    df["net_sales"] > df["category_avg_sales"]
).astype(int)

df[["category", "net_sales", "category_avg_sales", "above_category_average"]].head(2)

,category,net_sales,category_avg_sales,above_category_average
0,Toys,301.300,270.479839,1
1,Clothing,70.739,269.108226,0


---
## Section 13: Percentage Share Within a Group

Business question: What share of total category sales does each order represent?

In [35]:
# Step 1: total sales for each category (one value per row)
df["category_total_sales"] = (
    df.groupby("category")["net_sales"].transform("sum")
)

# Step 2: divide each row by its category total
df["category_sales_share"] = (
    df["net_sales"] / df["category_total_sales"]
)

df[["category", "net_sales", "category_total_sales", "category_sales_share"]].head(2)

,category,net_sales,category_total_sales,category_sales_share
0,Toys,301.300,8384.875,0.035934
1,Clothing,70.739,8342.355,0.008480


### ✏️ Exercise 8

Create a column country_sales_share : each order's share of its country's total net_sales .

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [36]:
# Solution
df["country_total_sales"] = (
    df.groupby("country")["net_sales"].transform("sum")
)

df["country_sales_share"] = (
    df["net_sales"] / df["country_total_sales"]
)

df[["country", "net_sales", "country_sales_share"]].head(2)

,country,net_sales,country_sales_share
0,UK,301.300,0.037150
1,USA,70.739,0.003879


---
## Section 14: Rank Within a Group

Business question: Within each country, which orders are the largest?

In [37]:
df.groupby("country")["net_sales"].rank(method="dense", ascending=False)

0      11.0
1      63.0
2      19.0
3      65.0
4       1.0
       ... 
235    12.0
236    13.0
237     8.0
238    17.0
239    23.0
Name: net_sales, Length: 240, dtype: float64

In [38]:
df["rank_within_country"] = (
    df
    .groupby("country")["net_sales"]
    .rank(method="dense", ascending=False)
)

df[["country", "order_id", "net_sales", "rank_within_country"]].sort_values(
    ["country", "rank_within_country"]
).head(2)

,country,order_id,net_sales,rank_within_country
46,Australia,ORD10011,824.820,1.0
45,Australia,ORD10035,427.735,2.0


To flag the top 25% within a group, use a quantile threshold:

In [42]:
df['net_sales'].quantile(0.75) # The 75th percentile

np.float64(379.77250000000004)

In [43]:
df.groupby('channel')['net_sales'].quantile(0.75)

channel
Marketplace    321.1475
Online         400.5050
Partner        281.6300
Retail         373.0300
Name: net_sales, dtype: float64

In [46]:
df.groupby('channel')['net_sales'].transform(lambda x: x.quantile(0.75))

0      321.1475
1      281.6300
2      321.1475
3      373.0300
4      400.5050
         ...   
235    373.0300
236    400.5050
237    321.1475
238    373.0300
239    321.1475
Name: net_sales, Length: 240, dtype: float64

In [ ]:
channel_75th = (
    df.groupby("channel")["net_sales"]
    .transform(lambda x: x.quantile(0.75))
)

df["top_25pct_in_channel"] = (df["net_sales"] >= channel_75th).astype(int)

df[["channel", "net_sales", "top_25pct_in_channel"]].head(10)

---
## Section 15: Cumulative Columns

Business question: For each customer, what is their running total of sales over time?

> ⚠️ **Sort Before Cumulating**
>
> Always sort the data before computing cumulative values, or the running totals will be meaningless.

In [48]:
# Ensure order_date is datetime
df["order_date"] = pd.to_datetime(df["order_date"])

# Sort by customer and date
df = df.sort_values(["customer_id", "order_date"])

# Running total of net_sales per customer
df["customer_running_sales"] = (
    # cumsum() returns one value per row (same length as the original DataFrame, unlike sum() which returns one value per group)
    df.groupby("customer_id")["net_sales"].cumsum()
)

# Sequential order number per customer (1st order, 2nd order, ...)
df["customer_order_number"] = (
    # df.groupby("customer_id").cumcount()  - assign a running count within each customer group
    df.groupby("customer_id").cumcount() + 1 
)

df[["customer_id", "order_date", "net_sales", 
    "customer_running_sales", "customer_order_number"]].head(15)

,customer_id,order_date,net_sales,customer_running_sales,customer_order_number
23,C1000,2025-01-08,302.8300,302.8300,1
148,C1000,2025-02-02,597.3400,900.1700,2
32,C1000,2025-02-12,149.5400,1049.7100,3
139,C1000,2025-04-04,602.7585,1652.4685,4
194,C1000,2025-05-07,259.6900,1912.1585,5
140,C1000,2025-05-24,-66.4610,1845.6975,6
166,C1001,2025-01-29,114.3170,114.3170,1
201,C1001,2025-01-30,-193.5540,-79.2370,2
181,C1001,2025-03-08,87.4500,8.2130,3
218,C1001,2025-03-10,787.4365,795.6495,4


---
## Section 16: Merging Tables

Often the columns you need come from different tables . We merge transactions with customers to get all the information in one place.

In [55]:
customers_advanced = pd.read_csv("customers_advanced.csv")
customers_advanced.head()

,customer_id,signup_date,loyalty_tier,marketing_opt_in
0,C1000,2024-09-22,Gold,Yes
1,C1001,2025-01-19,Gold,No
2,C1002,2024-06-09,Silver,Yes
3,C1003,2024-08-04,Silver,Yes
4,C1004,2025-01-26,Bronze,Yes


In [56]:
df.head()

,order_id,customer_id,order_date,country,channel,category,segment,payment_method,ship_method,quantity,unit_price,discount_rate,shipping_fee,status,satisfaction_score,discount_rate_clean,gross_sales,discount_amount,net_sales,discount_rate_pct,high_value_order,is_return_or_cancelled,has_discount,order_status_group,review_needed,vip_order,priority_review,risk_level,shipping_label,region,region_v2,revenue_after_return,country_avg_sales,above_country_average,category_avg_sales,above_category_average,category_total_sales,category_sales_share,country_total_sales,country_sales_share,rank_within_country,customer_running_sales,customer_order_number
23,ORD10099,C1000,2025-01-08,USA,Marketplace,Books,Small Business,Credit Card,Express,3,94.28,0.00,19.99,Cancelled,3.0,0.00,282.84,0.0000,302.8300,0,True,1,0,Problem Order,Yes,0,1,High Risk,Paid Shipping - High Value,North America,North America,0.0000,239.955954,1,251.664222,1,9059.912,0.033425,18236.6525,0.016606,23.0,302.8300,1
148,ORD10190,C1000,2025-02-02,UK,Retail,Books,Consumer,Credit Card,Standard,5,118.47,0.00,4.99,Returned,1.0,0.00,592.35,0.0000,597.3400,0,True,1,0,Problem Order,Yes,0,1,High Risk,Paid Shipping - High Value,International,International,0.0000,289.659554,1,251.664222,1,9059.912,0.065932,8110.4675,0.073651,2.0,900.1700,2
32,ORD10201,C1000,2025-02-12,USA,Online,Beauty,Consumer,Credit Card,Standard,1,141.55,NaN,7.99,Completed,4.0,0.00,141.55,0.0000,149.5400,0,False,0,0,Completed Order,No,0,0,Low Risk,Paid Shipping - High Value,North America,North America,149.5400,239.955954,0,260.138861,0,9364.999,0.015968,18236.6525,0.008200,48.0,1049.7100,3
139,ORD10158,C1000,2025-04-04,Germany,Retail,Beauty,Consumer,Credit Card,Standard,7,89.89,0.05,4.99,Completed,NaN,0.05,629.23,31.4615,602.7585,5,True,0,1,Completed Order,No,0,1,Low Risk,Paid Shipping - High Value,International,International,602.7585,336.902087,1,260.138861,1,9364.999,0.064363,7748.7480,0.077788,5.0,1652.4685,4
194,ORD10131,C1000,2025-05-07,USA,Online,Home,Small Business,Credit Card,Standard,5,50.34,0.00,7.99,Cancelled,4.0,0.00,251.70,0.0000,259.6900,0,True,1,0,Problem Order,Yes,0,1,High Risk,Paid Shipping - High Value,North America,North America,0.0000,239.955954,1,317.986838,0,11765.513,0.022072,18236.6525,0.014240,26.0,1912.1585,5


In [57]:
# Python — left merge with indicator
tx_all = df.merge(
    customers,
    on="customer_id",
    how="left",
    indicator=True
)

tx_all.head(2)

,order_id,customer_id,order_date,country,channel,category,segment,payment_method,ship_method,quantity,unit_price,discount_rate,shipping_fee,status,satisfaction_score,discount_rate_clean,gross_sales,discount_amount,net_sales,discount_rate_pct,high_value_order,is_return_or_cancelled,has_discount,order_status_group,review_needed,vip_order,priority_review,risk_level,shipping_label,region,region_v2,revenue_after_return,country_avg_sales,above_country_average,category_avg_sales,above_category_average,category_total_sales,category_sales_share,country_total_sales,country_sales_share,rank_within_country,customer_running_sales,customer_order_number,signup_date,loyalty_tier,marketing_opt_in,_merge
0,ORD10099,C1000,2025-01-08,USA,Marketplace,Books,Small Business,Credit Card,Express,3,94.28,0.0,19.99,Cancelled,3.0,0.0,282.84,0.0,302.83,0,True,1,0,Problem Order,Yes,0,1,High Risk,Paid Shipping - High Value,North America,North America,0.0,239.955954,1,251.664222,1,9059.912,0.033425,18236.6525,0.016606,23.0,302.83,1,2024-09-22,Gold,Yes,both
1,ORD10190,C1000,2025-02-02,UK,Retail,Books,Consumer,Credit Card,Standard,5,118.47,0.0,4.99,Returned,1.0,0.0,592.35,0.0,597.34,0,True,1,0,Problem Order,Yes,0,1,High Risk,Paid Shipping - High Value,International,International,0.0,289.659554,1,251.664222,1,9059.912,0.065932,8110.4675,0.073651,2.0,900.17,2,2024-09-22,Gold,Yes,both


> ℹ️ **What is _merge?**
>
> indicator=True adds a _merge column showing how each row matched: "both" → matched in both tables ✓ "left_only" → transaction has no matching customer ⚠ "right_only" → customer with no transactions

In [58]:
# Check merge quality
tx_all["_merge"].value_counts()

# Once satisfied, remove the helper column
tx_all = tx_all.drop(columns=["_merge"])

---
## Section 17: Columns from Merged Data

Now that we have customer information (like loyalty_tier ) in the same row as transaction data, we can create columns that combine both.

In [60]:
# Priority order: loyalty tier is Gold/Platinum AND net_sales > 200
tx_all["priority_customer_order"] = (
    tx_all["loyalty_tier"].isin(["Gold", "Platinum"])
    & (tx_all["net_sales"] > 200)
).astype(int)

tx_all[["customer_id", "loyalty_tier", "net_sales", "priority_customer_order"]].head(5)

,customer_id,loyalty_tier,net_sales,priority_customer_order
0,C1000,Gold,302.8300,1
1,C1000,Gold,597.3400,1
2,C1000,Gold,149.5400,0
3,C1000,Gold,602.7585,1
4,C1000,Gold,259.6900,1


### ✏️ Exercise 9

Create a column called student_discount_flag . Set it to 1 when:
segment == "Student" , AND discount_amount > 0
Otherwise 0 .

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [62]:
# Solution
tx_all["student_discount_flag"] = (
    (tx_all["segment"] == "Student")
    & (tx_all["discount_amount"] > 0)
).astype(int)

tx_all[["segment", "discount_amount", "student_discount_flag"]].head(2)

,segment,discount_amount,student_discount_flag
0,Small Business,0.0,0
1,Consumer,0.0,0


---
## Section 18: Dummy Variables

Dummy variables convert a categorical column into multiple 0/1 columns — one per category. Used in machine learning and regression.

In [63]:
# Python — create dummies for channel
channel_dummies = pd.get_dummies(
    tx_all["channel"],
    prefix="channel",
    dummy_na=True,   # also flag missing values
    dtype=int
)

channel_dummies.head()

,channel_Marketplace,channel_Online,channel_Partner,channel_Retail,channel_nan
0,1,0,0,0,0
1,0,0,0,1,0
2,0,1,0,0,0
3,0,0,0,1,0
4,0,1,0,0,0


In [64]:
# Python — combine with original DataFrame
# pd.concat([tx_all, channel_dummies], axis=1)
tx_all = pd.concat([tx_all, channel_dummies], axis=1)

# Check the new columns
tx_all.filter(regex="^channel_").head()

,channel_Marketplace,channel_Online,channel_Partner,channel_Retail,channel_nan
0,1,0,0,0,0
1,0,0,0,1,0
2,0,1,0,0,0
3,0,0,0,1,0
4,0,1,0,0,0


> 💡 **The Dummy Variable Trap**
>
> For regression models, drop one category to avoid perfect multicollinearity: pd.get_dummies(df["channel"], prefix="channel", drop_first=True) For tree-based models (Random Forest, XGBoost), keeping all dummies is fine.

### ✏️ Exercise 10

Create dummy columns for loyalty_tier using prefix "tier" . Then combine them with tx_all .

In [65]:
# Your code here

**Solution** (try it yourself first!)

In [68]:
pd.get_dummies(tx_all['loyalty_tier'])

,Bronze,Gold,Platinum,Silver
0,False,True,False,False
1,False,True,False,False
2,False,True,False,False
3,False,True,False,False
4,False,True,False,False
...,...,...,...,...
235,False,False,False,True
236,True,False,False,False
237,True,False,False,False
238,True,False,False,False


In [66]:
# Solution
tier_dummies = pd.get_dummies(
    tx_all["loyalty_tier"],
    prefix="tier",
    dtype=int
)

tx_all = pd.concat([tx_all, tier_dummies], axis=1)

tx_all.filter(regex="^tier_").head()

,tier_Bronze,tier_Gold,tier_Platinum,tier_Silver
0,0,1,0,0
1,0,1,0,0
2,0,1,0,0
3,0,1,0,0
4,0,1,0,0


---
## Section 19: replace() and map()

replace() is useful for changing specific values. map() converts categories to labels or scores using a dictionary.

In [ ]:
# replace(): rename specific values
df["status_clean"] = df["status"].replace({
    "Completed": "complete",
    "Returned":  "problem",
    "Cancelled": "problem"
})

# map(): convert categories to numeric scores
tier_score_map = {
    "Bronze": 1,
    "Silver": 2,
    "Gold":   3,
    "Platinum": 4
}
tx_all["loyalty_score"] = tx_all["loyalty_tier"].map(tier_score_map)

tx_all[["loyalty_tier", "loyalty_score"]].head(10)

---
## Section 20: Delete Columns — .drop(), del, pop()

There are three ways to delete columns. They each have a different use case.

### Method 1 .drop() — recommended for beginners

In [ ]:
# Drop one column
df = df.drop(columns=["region_v2"])

# Drop multiple columns
df = df.drop(columns=["country_avg_sales", "country_total_sales"], errors="ignore")

# errors="ignore" prevents an error if the column doesn't exist

### Method 2 del — direct deletion of one column

In [ ]:
df["temp_col"] = "temporary"
del df["temp_col"]

# Modifies the DataFrame directly (no reassignment needed)
# Cannot delete multiple columns in one line

### Method 3 pop() — remove and save

In [ ]:
df["pop_example"] = df["net_sales"] * 10

# Remove from DataFrame AND save as a separate Series
saved = df.pop("pop_example")

saved.head()           # still available
"pop_example" in df.columns   # False — removed from df

> 💡 **Recommended Style for Beginners**
>
> Use df = df.drop(columns=["col"]) most of the time — it is explicit, works for multiple columns, and plays well in pipelines. Avoid inplace=True — it makes code harder to read and chain.

### ✏️ Exercise 11

Drop the columns category_avg_sales and category_total_sales from df . Use errors="ignore" .

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [ ]:
# Solution
df = df.drop(
    columns=["category_avg_sales", "category_total_sales"],
    errors="ignore"
)

df.head()

---
## Section 21: Drop Rows — Filter, dropna()

### Method 1 Filter rows with .loc (safest)

Instead of deleting, create a new DataFrame with only the rows you want to keep.

In [ ]:
# Keep only non-return orders
non_returns = df.loc[df["is_return_or_cancelled"] == 0].copy()

non_returns.head()

### Method 2 dropna() — remove rows with missing values

In [ ]:
# Drop rows where country is missing
df_clean = df.dropna(subset=["country"])

# Drop rows where ANY column has missing values
df_complete = df.dropna()

### Method 3 dropna(axis=1) — drop columns with too many missing values

In [ ]:
# Keep only columns with at least 50 non-missing values
df_cleaned = df.dropna(axis=1, thresh=50)

### ✏️ Exercise 12

Create a DataFrame called rows_with_country that keeps only rows where country is not missing. Use dropna() .

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [ ]:
# Solution
rows_with_country = df.dropna(subset=["country"])

rows_with_country.head()

---
## Section 22: Select Columns to Keep

When creating a final clean dataset, it is often easier to list the columns you want to keep rather than dropping each unwanted one.

In [ ]:
final_columns = [
    "order_id",
    "customer_id",
    "order_date",
    "country",
    "category",
    "channel",
    "net_sales",
    "order_size",
    "risk_level"
]

clean_view = df[final_columns].copy()

clean_view.head()

### ✏️ Exercise 13

Create a DataFrame called model_data keeping only: net_sales , discount_rate_clean , shipping_fee , is_return_or_cancelled , channel , category .

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [ ]:
# Solution
model_data = df[[
    "net_sales",
    "discount_rate_clean",
    "shipping_fee",
    "is_return_or_cancelled",
    "channel",
    "category"
]].copy()

model_data.head()

---
## Section 23: Rename Columns

After creating or cleaning columns, we may want to rename them for clarity.

In [ ]:
renamed = model_data.rename(columns={
    "net_sales":           "sales",
    "discount_rate_clean": "discount_pct",
    "shipping_fee":        "shipping"
})

renamed.head()

---
## Section 24: Mini Case Study — Executive Dashboard

This combines many skills: groupby , agg() , .assign() , and sort_values() .

Business question: For each country, summarize performance for executives.

In [ ]:
executive_country = (
    tx_all
    .groupby("country")
    .agg(
        total_sales      =("net_sales",    "sum"),
        transactions     =("order_id",     "count"),
        unique_customers =("customer_id",  "nunique"),
        avg_sales        =("net_sales",    "mean"),
        return_rate      =("is_return_or_cancelled", "mean")
    )
    .assign(
        sales_per_customer=lambda x: x["total_sales"] / x["unique_customers"]
    )
    .sort_values("total_sales", ascending=False)
)

executive_country

> ℹ️ **What does .assign() do here?**
>
> After aggregation, the result is a new DataFrame. .assign() adds a column to that new DataFrame . The lambda x: refers to the DataFrame produced by the previous .agg() step.

### Classroom Mini-Case: Clean a Dataset for Reporting

In [ ]:
# Python — full workflow: create → use → clean up → keep only reporting columns
report = tx_all.copy()

# Step 1: Create helper columns
report["helper_high_discount"] = (
    report["discount_rate_clean"] >= 0.15
).astype(int)

# Step 2: Create final reporting column
report["report_flag"] = np.where(
    (report["helper_high_discount"] == 1) | (report["is_return_or_cancelled"] == 1),
    "Review",
    "OK"
)

# Step 3: Drop helper columns
report = report.drop(columns=["helper_high_discount"])

# Step 4: Keep only reporting columns
report = report[["order_id", "customer_id", "country", 
                 "category", "channel", "net_sales", "report_flag"]]

report.head()

### ✏️ Exercise 14 — Full Workflow

Create a DataFrame called student_report . Follow these steps:
1 Start from tx_all.copy() 2 Create helper column helper_large_order = 1 if net_sales >= 150 3 Create student_flag : "Important" if helper_large_order==1 AND segment=="Student" , else "Normal" 4 Drop helper_large_order 5 Keep only: order_id , customer_id , segment , net_sales , student_flag

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [ ]:
# Solution
student_report = tx_all.copy()

student_report["helper_large_order"] = (
    student_report["net_sales"] >= 150
).astype(int)

student_report["student_flag"] = np.where(
    (student_report["helper_large_order"] == 1)
    & (student_report["segment"] == "Student"),
    "Important",
    "Normal"
)

student_report = student_report.drop(columns=["helper_large_order"])

student_report = student_report[[
    "order_id", "customer_id", "segment", "net_sales", "student_flag"
]]

student_report.head()

---
## Section 25: Additional Classroom Practice Exercises

These exercises are for additional in-class practice. Start with a fresh copy:

In [ ]:
# Python — setup for practice exercises
practice = transactions.copy()
practice["discount_rate_clean"] = practice["discount_rate"].fillna(0)
practice["gross_sales"] = practice["quantity"] * practice["unit_price"]
practice["net_sales"] = (
    practice["gross_sales"] * (1 - practice["discount_rate_clean"])
    + practice["shipping_fee"]
)

### ✏️ Practice 1

Create review_needed : default "No" , change to "Yes" for Returned/Cancelled orders.

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [ ]:
# Solution
practice["review_needed"] = "No"
practice.loc[practice["status"].isin(["Returned", "Cancelled"]), "review_needed"] = "Yes"
practice[["order_id", "status", "review_needed"]].head(10)

### ✏️ Practice 2

Create high_shipping_flag : 1 when shipping_fee > 10 , otherwise 0 .

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [ ]:
# Solution
practice["high_shipping_flag"] = 0
practice.loc[practice["shipping_fee"] > 10, "high_shipping_flag"] = 1
practice[["order_id", "shipping_fee", "high_shipping_flag"]].head(10)

### ✏️ Practice 3

Create sales_note :
"High value completed order" if net_sales >= 300 and status == "Completed" "Problem order" if status is Returned or Cancelled "Regular order" otherwise

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [ ]:
# Solution
practice["sales_note"] = "Regular order"

practice.loc[
    (practice["net_sales"] >= 300) & (practice["status"] == "Completed"),
    "sales_note"
] = "High value completed order"

practice.loc[
    practice["status"].isin(["Returned", "Cancelled"]),
    "sales_note"
] = "Problem order"

practice[["status", "net_sales", "sales_note"]].head(15)

### ✏️ Practice 4

Update two columns at once for cancelled orders: set net_sales to 0 and sales_note to "Cancelled order" .

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [ ]:
# Solution
practice.loc[
    practice["status"] == "Cancelled",
    ["net_sales", "sales_note"]
] = [0, "Cancelled order"]

practice.loc[
    practice["status"] == "Cancelled",
    ["order_id", "status", "net_sales", "sales_note"]
].head(10)

### ✏️ Practice 5

Fix missing values: replace missing country with "Unknown" and missing customer_segment with "Unassigned" using .loc . Then verify no missing values remain.

In [ ]:
# Your code here

**Solution** (try it yourself first!)

In [ ]:
# Solution
practice.loc[practice["country"].isna(), "country"] = "Unknown"
practice.loc[practice["customer_segment"].isna(), "customer_segment"] = "Unassigned"

practice[["country", "customer_segment"]].isna().sum()

---
## Section 26: Common Beginner Mistakes

### Mistake 1 Chained Assignment

In [ ]:
# ❌ Wrong
# May silently not update the original DataFrame
df[df["net_sales"] > 100]["flag"] = 1

In [ ]:
# ✅ Correct
df.loc[df["net_sales"] > 100, "flag"] = 1

### Mistake 2 Missing Parentheses Around Conditions

In [ ]:
# ❌ Wrong — operator precedence issue
df["net_sales"] > 100 & df["is_return"] == 1

In [ ]:
# ✅ Correct — each condition in its own parentheses
(df["net_sales"] > 100) & (df["is_return"] == 1)

### Mistake 3 Using apply() for Everything

In [ ]:
# ❌ Slow and unnecessary
df["doubled"] = df["net_sales"].apply(lambda x: x * 2)

In [ ]:
# ✅ Faster vectorized operation
df["doubled"] = df["net_sales"] * 2

### Mistake 4 Forgetting to Reassign After .drop()

In [ ]:
# ❌ Wrong — df is unchanged
df.drop(columns=["temp_col"])  # returns a new DataFrame, doesn't modify df

In [ ]:
# ✅ Correct — reassign the result
df = df.drop(columns=["temp_col"])

### Mistake 5 Not Sorting Before cumsum()

In [ ]:
# ❌ Wrong — cumulative totals are meaningless without sorting
df["running_total"] = df.groupby("customer_id")["net_sales"].cumsum()

In [ ]:
# ✅ Correct — sort first
df = df.sort_values(["customer_id", "order_date"])
df["running_total"] = df.groupby("customer_id")["net_sales"].cumsum()

---
## Section 27: Quick Reference Cheat Sheet

### Creating Columns

| Task | Code |
| --- | --- |
| Arithmetic | df["new"] = df["a"] + df["b"] |
| Boolean flag | df["flag"] = (df["col"] > 100).astype(int) |
| If/else (two choices) | df["col"] = np.where(condition, "yes", "no") |
| Multiple categories | df["col"] = np.select(conditions, choices, default=...) |
| Custom logic (multi-column) | df["col"] = df.apply(my_func, axis=1) |
| Simple transform | df["col"] = df["x"].apply(lambda x: x*2) |
| Pipeline style | df = df.assign(new_col=lambda x: x["a"] / x["b"]) |

### Updating Columns

| Task | Code |
| --- | --- |
| Update rows matching condition | df.loc[condition, "col"] = value |
| Update multiple columns | df.loc[cond, ["col1","col2"]] = [v1, v2] |
| Replace specific values | df["col"].replace({"old": "new"}) |
| Map values via dictionary | df["col"].map({"A": 1, "B": 2}) |

### Group-Based Columns

| Task | Code |
| --- | --- |
| Group average per row | df.groupby("g")["col"].transform("mean") |
| Group total per row | df.groupby("g")["col"].transform("sum") |
| Rank within group | df.groupby("g")["col"].rank(method="dense", ascending=False) |
| Running total within group | df.groupby("g")["col"].cumsum() |
| Order number within group | df.groupby("g").cumcount() + 1 |

### Deleting & Cleaning

| Task | Code |
| --- | --- |
| Drop one column | df = df.drop(columns=["col"]) |
| Drop multiple columns | df = df.drop(columns=["c1","c2"], errors="ignore") |
| Delete directly | del df["col"] |
| Remove and save column | saved = df.pop("col") |
| Drop rows (missing values) | df.dropna(subset=["col"]) |
| Select columns to keep | df = df[["col1", "col2", "col3"]] |
| Rename columns | df.rename(columns={"old": "new"}) |

> 💡 **Golden Rule for Beginners**
>
> Always work on a .copy() of the original data. This means you can always start fresh without reloading the file.